# ReF-DIM Testing Notebook

This notebook contains the complete testing script for ReF-DIM model.

## Setup Instructions

1. Run the first cell to check/install dependencies
2. Configure paths in the configuration cell
3. Load your trained model
4. Run inference on images


## Step 1: Install Required Packages

Check and install dependencies if needed.


In [ ]:
# Install required packages (if needed)
# Uncomment the following lines to install packages

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install pillow

# Check installed packages
required_packages = ['torch', 'torchvision', 'PIL']
missing_packages = []

for package in required_packages:
    try:
        if package == 'PIL':
            __import__('PIL')
        else:
            __import__(package)
        print(f"✓ {package} is installed")
    except ImportError:
        print(f"✗ {package} is NOT installed")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
    print("Please uncomment and run the pip install commands above.")
else:
    print("\n✅ All required packages are installed!")


## Step 2: Import Libraries


In [ ]:
import torch
import torchvision
import os
import time
from PIL import Image
from torchvision.transforms import Compose, ToTensor
import glob

from model.DIM import DIM


## Step 3: Configuration

Set your paths and model parameters here.


In [ ]:
# Test Configuration
class Config:
    # Model parameters (must match training configuration)
    model_range = 6  # Number of enhancement stages
    c_hidden = 32    # Hidden channels
    
    # Paths
    model_path = r'snapshot/best.pth'  # Path to trained model
    input_folder = r'path/to/test/images'  # Path to input images folder
    output_folder = r'results'  # Path to save enhanced images
    
    # Device
    device_id = 0  # GPU device ID (if using GPU)

config = Config()


## Step 4: Verify Paths and Setup


In [ ]:
# Verify model path exists
if os.path.exists(config.model_path):
    print(f"✓ Model path is valid: {config.model_path}")
else:
    print(f"✗ Model path does not exist: {config.model_path}")
    print("Please update config.model_path in the configuration cell above")

# Verify input folder exists
if os.path.exists(config.input_folder):
    print(f"✓ Input folder is valid: {config.input_folder}")
    # Count image files
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp']
    image_files = [f for f in os.listdir(config.input_folder) 
                   if any(f.lower().endswith(ext) for ext in image_extensions)]
    print(f"  Found {len(image_files)} image files")
else:
    print(f"✗ Input folder does not exist: {config.input_folder}")
    print("Please update config.input_folder in the configuration cell above")

# Create output folder if it doesn't exist
os.makedirs(config.output_folder, exist_ok=True)
print(f"✓ Output folder: {config.output_folder}")

# Check device availability
device = torch.device(f"cuda:{config.device_id}" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(config.device_id)}")
    print(f"  Memory: {torch.cuda.get_device_properties(config.device_id).total_memory / 1024**3:.2f} GB")


## Step 5: Define Image Processing Function


In [ ]:
def lowlight(image_path, net, result_path, device):
    """
    Process a single low-light image using the DIM model.

    Args:
        image_path (str): Path to input image.
        net (nn.Module): Pre-loaded DIM model.
        result_path (str): Path to output image.
        device (torch.device): Device to run the model on.
    """
    # Define image preprocessing pipeline
    transform = Compose([
        ToTensor()  # Convert PIL image to C×H×W Tensor and normalize to [0, 1]
    ])

    # Load and preprocess image
    data_lowlight = Image.open(image_path).convert('RGB')
    data_lowlight = transform(data_lowlight)  # Apply transform
    data_lowlight = data_lowlight.unsqueeze(0).to(device)  # Add batch dimension

    # Perform inference
    with torch.no_grad():
        enhanced_images = net(data_lowlight)

    # If model returns a list of outputs, take the last (most enhanced) one
    if isinstance(enhanced_images, (list, tuple)):
        enhanced_img = enhanced_images[-1].squeeze(0)
    else:
        enhanced_img = enhanced_images.squeeze(0)

    # Save enhanced image
    # Make sure output directory exists
    os.makedirs(os.path.dirname(result_path) if os.path.dirname(result_path) else '.', exist_ok=True)
    torchvision.utils.save_image(enhanced_img, result_path)

    # Clean up
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Image processing function defined ✓")


## Step 6: Load Model


In [ ]:
# Initialize model with same parameters as training
net = DIM(c1=3, c_hidden=config.c_hidden, range=config.model_range).to(device)

# Load trained model weights
print(f"Loading model from: {config.model_path}")
try:
    if torch.cuda.is_available():
        net.load_state_dict(torch.load(config.model_path))
    else:
        net.load_state_dict(torch.load(config.model_path, map_location='cpu'))
    
    net.eval()  # Set to evaluation mode
    print("✓ Model loaded successfully!")
    
    # Print model info
    total_params = sum(p.numel() for p in net.parameters())
    trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
except Exception as e:
    print(f"✗ Error loading model: {e}")
    print("Please check the model path and ensure it matches the model architecture.")


## Step 7: Get Image Files List


In [ ]:
# Get all image files from input folder
def is_image_file(filename):
    """Check if file is an image based on extension"""
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.PNG', '.JPG', '.JPEG', '.BMP']
    return any(filename.endswith(ext) for ext in image_extensions)

image_files = [
    os.path.join(config.input_folder, f) 
    for f in os.listdir(config.input_folder) 
    if is_image_file(f)
]

print(f"Found {len(image_files)} image files to process:")
for i, img_path in enumerate(image_files[:10]):  # Show first 10
    print(f"  {i+1}. {os.path.basename(img_path)}")
if len(image_files) > 10:
    print(f"  ... and {len(image_files) - 10} more")


## Step 8: Process Images

Run inference on all images in the input folder.


In [ ]:
# Process all images
print(f"Starting image processing...")
print(f"Input folder: {config.input_folder}")
print(f"Output folder: {config.output_folder}")
print(f"Total images: {len(image_files)}\n")

total_time = 0
processed_count = 0

for idx, image_path in enumerate(image_files, 1):
    # Generate output path
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    result_path = os.path.join(config.output_folder, f"{base_name}.png")
    
    try:
        # Process image
        start_time = time.time()
        lowlight(image_path, net, result_path, device)
        end_time = time.time()
        elapsed_time = end_time - start_time
        total_time += elapsed_time
        processed_count += 1
        
        print(f"[{idx}/{len(image_files)}] ✓ {os.path.basename(image_path)} -> {os.path.basename(result_path)} ({elapsed_time:.4f}s)")
        
    except Exception as e:
        print(f"[{idx}/{len(image_files)}] ✗ Error processing {os.path.basename(image_path)}: {e}")

# Summary
print(f"\n{'='*50}")
print(f"Processing completed!")
print(f"  Successfully processed: {processed_count}/{len(image_files)} images")
if processed_count > 0:
    print(f"  Total time: {total_time:.2f}s")
    print(f"  Average time per image: {total_time/processed_count:.4f}s")
print(f"  Results saved to: {config.output_folder}")
print(f"{'='*50}")
